# LLM 실습 문제 - 풀이

**GPT from scratch** 핵심 개념과 상세 풀이.

흐름: Tokenization → Sliding Window Dataset → Token/Positional Embedding → Causal Self-Attention → Multi-Head Attention → Transformer Block (LayerNorm, FFN, Residual) → GPT 아키텍처 → Pretraining (Loss/Training Loop/Generation) → Fine-tuning (LoRA / Classification / Instruction / DPO)

## Setup

In [ ]:
# [코드]
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 핵심 개념 1. BPE (Byte Pair Encoding) Tokenizer

**역할**: 사람이 읽는 문자열(text) ↔ 모델이 이해하는 정수 ID 시퀀스(token) 변환.

- `tiktoken.get_encoding("gpt2")`: GPT-2가 쓰는 BPE 토크나이저 로드.
- `tokenizer.encode(text, allowed_special={...})` → **List[int]** (토큰 ID 시퀀스)
- `tokenizer.decode(integers)` → **str**
- `allowed_special={"<|endoftext|>"}`: 특수 토큰을 명시적으로 허용해야 `<|endoftext|>`가 하나의 토큰으로 인코딩 됨(기본값은 보안상 에러 발생).

**왜 BPE?**: 단어 단위(word)는 OOV 문제, 문자 단위(char)는 시퀀스가 너무 김. BPE는 자주 쓰는 서브워드를 병합해 둘의 장점을 취함.

### 문제 1. BPE Tokenizer encode / decode

In [ ]:
# [코드]
tokenizer = tiktoken.get_encoding("gpt2")

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)

integers = tokenizer.### encode ### 작성 필요 ###(text, allowed_special={"<|endoftext|>"})
print(integers)

strings = tokenizer.### decode ### 작성 필요 ###(integers)
print(strings)

## 핵심 개념 2. Next-Token Prediction (GPT 학습 방식)

GPT는 **이전 토큰들로 다음 토큰을 예측**하는 auto-regressive 모델. 따라서 정답(y)은 입력(x)을 **1칸씩 오른쪽으로 밀어낸** 시퀀스.

- 입력: `x = tokens[0:context_size]`
- 정답: `y = tokens[1:context_size+1]` ← 한 칸 shift
- 예시: `x=[A, B, C, D]` → `y=[B, C, D, E]`
  - 위치 0에서 A를 보고 B 예측
  - 위치 1에서 A,B를 보고 C 예측
  - ...

**핵심**: x와 y는 **같은 길이**이고, y는 x를 1칸 shift한 것. 이것이 "언어 모델"의 학습 신호.

### 문제 2. Next-Token Prediction: 입력(x) / 정답(y) 쌍 만들기

In [ ]:
# [코드]
enc_text = tokenizer.encode(raw_text)
enc_sample = enc_text[50:]

context_size = 4
x = enc_sample[:context_size]
y = enc_sample[### 1 ### 작성 필요 ###:context_size+### 1 ### 작성 필요 ###]

print(f"x: {x}")
print(f"y:      {y}")

## 핵심 개념 3. GPT Dataset: Sliding Window

긴 텍스트를 `max_length` 크기의 윈도우로 잘라 **(input, target) 쌍**을 대량 생성.

- `stride`: 윈도우가 한 번에 얼마나 이동하는지 (stride=max_length이면 overlap 없음, stride<max_length이면 overlap 있음).
- `i` 번째 샘플: `input_chunk = tokens[i : i+max_length]`, `target_chunk = tokens[i+1 : i+max_length+1]`
- target은 input을 **1 shift** 한 것 (Next-Token Prediction 원리).

**__getitem__**: 텐서로 저장해둔 `input_ids[idx]`, `target_ids[idx]`를 그대로 반환.

**Dataset 3-필수 메서드**: `__init__` (데이터 로딩/전처리), `__len__`, `__getitem__`.

### 문제 3. GPTDataset: sliding window로 (input, target) 생성

In [ ]:
# [코드]
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "토큰화된 입력의 개수는 적어도 max_length+1과 같아야 합니다."

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[### i ### 작성 필요 ### : ### i ### 작성 필요 ### + max_length]
            target_chunk = token_ids[### i ### 작성 필요 ### + ### 1 ### 작성 필요 ### : ### i ### 작성 필요 ### + max_length + ### 1 ### 작성 필요 ###]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[### idx ### 작성 필요 ###], self.target_ids[### idx ### 작성 필요 ###]

## 핵심 개념 4. Token Embedding + Positional Embedding

Transformer는 **순서 정보가 없는** 구조(set-like)이므로 위치 정보를 별도로 더해줘야 함.

- **Token Embedding**: `nn.Embedding(vocab_size, emb_dim)` — 어떤 단어(ID)인지.
  - 입력: 정수 ID 텐서 `(batch, seq_len)` → 출력: `(batch, seq_len, emb_dim)`
- **Positional Embedding**: `nn.Embedding(context_length, emb_dim)` — 시퀀스 내 몇 번째 위치인지.
  - 입력: `torch.arange(seq_len)` — `[0, 1, 2, ..., seq_len-1]`
  - 출력: `(seq_len, emb_dim)` → broadcast로 모든 배치에 동일한 위치 정보 더함.
- 최종 입력: `input_embeddings = token_embeddings + pos_embeddings`

**주의**: Positional Embedding의 **첫 인자는 `vocab_size`가 아니라 `max_length(=context_length)`**. 위치 개수만큼만 있으면 됨.

### 문제 4. Token Embedding + Positional Embedding

In [ ]:
# [코드]
vocab_size = 50257
output_dim = 256
max_length = 4

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

pos_embedding_layer = torch.nn.Embedding(### max_length ### 작성 필요 ###, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(### max_length ### 작성 필요 ###))
print(pos_embeddings.shape)

input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

## 핵심 개념 5. Causal (Masked) Self-Attention

**Self-Attention 핵심 수식**:

$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V $$

**단계별 풀이**:
1. **Q, K, V 생성**: 입력 x에 각각 `nn.Linear(d_in, d_out)` 적용 → Q, K, V 동일 shape `(b, num_tokens, d_out)`.
2. **Attention score**: `queries @ keys.transpose(1, 2)` — 토큰 i가 토큰 j를 얼마나 봐야 하는지. Shape `(b, num_tokens, num_tokens)`.
3. **Causal Mask**: GPT는 미래를 보면 안 됨(치팅 방지). 상삼각(diagonal=1) 부분을 `-inf`로 채워 softmax 후 0이 되게 함.
4. **Scaled Softmax**: `softmax(scores / √d_k)` — √d_k 스케일링은 내적 값이 커질수록 gradient 소실되는 것을 방지.
5. **Weighted sum**: `attn_weights @ values` → context vector.

**왜 `-torch.inf`?**: softmax(-∞) = 0 이므로 가려진 위치는 attention 0.

**register_buffer**: 학습은 안 되지만 `.to(device)` 시 같이 이동하는 상수 텐서 저장용 (mask 용도).

### 문제 5. Causal (Masked) Self-Attention

In [ ]:
# [코드]
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()

        self.W_query = nn.### Linear ### 작성 필요 ###(### d_in ### 작성 필요 ###, ### d_out ### 작성 필요 ###, bias=qkv_bias)
        self.W_key   = nn.### Linear ### 작성 필요 ###(### d_in ### 작성 필요 ###, ### d_out ### 작성 필요 ###, bias=qkv_bias)
        self.W_value = nn.### Linear ### 작성 필요 ###(### d_in ### 작성 필요 ###, ### d_out ### 작성 필요 ###, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.### W_key ### 작성 필요 ###(### x ### 작성 필요 ###)
        queries = self.### W_query ### 작성 필요 ###(### x ### 작성 필요 ###)
        values = self.### W_value ### 작성 필요 ###(### x ### 작성 필요 ###)

        attn_scores = ### queries ### 작성 필요 ### @ ### keys ### 작성 필요 ###.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens],
            ### -torch.inf ### 작성 필요 ###
        )
        attn_weights = torch.### softmax ### 작성 필요 ###(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ ### values ### 작성 필요 ###

        return context_vec

## 핵심 개념 6. Multi-Head Attention (효율적 구현)

**아이디어**: 여러 개의 attention을 병렬로 계산 → 각 head가 서로 다른 "관점"을 학습. `d_out`을 `num_heads` 조각으로 쪼개어 각 head에 `head_dim = d_out // num_heads` 만큼 배정.

**reshape / transpose 4단계** (핵심 트릭):
1. `(b, num_tokens, d_out)` ← Linear 출력
2. `.view(b, num_tokens, num_heads, head_dim)` ← head 차원 추가
3. `.transpose(1, 2)` → `(b, num_heads, num_tokens, head_dim)` ← head를 배치처럼 취급하면 병렬 계산 가능

**Attention score (4차원)**: `queries @ keys.transpose(2, 3)` → `(b, num_heads, num_tokens, num_tokens)`. 마지막 두 차원에서 K의 전치가 필요하므로 `transpose(2, 3)`.

**Head 결합**:
1. `context_vec.transpose(1, 2)` → `(b, num_tokens, num_heads, head_dim)`
2. `.contiguous().view(b, num_tokens, d_out)` ← head 차원들을 하나로 합침 (`.contiguous()`는 메모리 연속성 보장, view 호출에 필요).
3. `out_proj(Linear)` ← 헤드 간 정보 혼합.

**mask 처리**: `self.mask.bool()[:num_tokens, :num_tokens]` — 실제 시퀀스 길이만큼만 슬라이싱.

### 문제 6. Multi-Head Attention (head 분할/결합)

In [ ]:
# [코드]
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = ### d_out // num_heads ### 작성 필요 ###

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = ### keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2) ### 작성 필요 ###
        values = ### values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2) ### 작성 필요 ###
        queries = ### queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2) ### 작성 필요 ###

        attn_scores = ### queries @ keys.transpose(2, 3) ### 작성 필요 ###

        mask_bool = self.### mask.bool()[:num_tokens, :num_tokens] ### 작성 필요 ###

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = ### torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1) ### 작성 필요 ###
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = ### context_vec.contiguous().view(b, num_tokens, self.d_out) ### 작성 필요 ###
        context_vec = self.out_proj(context_vec)

        return context_vec

## 핵심 개념 7. Layer Normalization

**목적**: 학습 안정화. 각 샘플 내에서 **feature 차원(dim=-1)** 기준으로 평균 0, 분산 1로 정규화.

**수식**:

$$ \text{LN}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta $$

- `mean = x.mean(dim=-1, keepdim=True)` — **feature 차원만** 평균 (batch/seq는 각자).
- `var = x.var(dim=-1, keepdim=True, unbiased=False)` — biased variance (Transformer 관례).
- `keepdim=True` 필요: broadcasting을 위해 차원 유지.
- `+ self.eps`: 0으로 나누는 것 방지.
- **학습 파라미터**: `scale(γ)` 와 `shift(β)` — 정규화 후 다시 원하는 분포로 학습.

**vs. BatchNorm**: BatchNorm은 batch 축 평균, LayerNorm은 feature 축 평균. NLP처럼 시퀀스 길이가 가변적인 경우 LayerNorm이 더 적합.

### 문제 7. LayerNorm

In [ ]:
# [코드]
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = ### x.mean(dim=-1, keepdim=True) ### 작성 필요 ###
        var = ### x.var(dim=-1, keepdim=True, unbiased=False) ### 작성 필요 ###
        norm_x = (x - ### mean ### 작성 필요 ###) / torch.sqrt(### var ### 작성 필요 ### + self.eps)
        return self.### scale ### 작성 필요 ### * norm_x + self.### shift ### 작성 필요 ###

## 핵심 개념 8. FeedForward (Position-wise MLP)

**GPT-2 표준 구조**: 확장(4배) → 비선형 → 축소.

1. `Linear(emb_dim, 4 * emb_dim)` — feature 공간을 4배 확장 (표현력 증가)
2. `GELU()` — 비선형 활성화 (ReLU보다 매끄러움, GPT-2 표준)
3. `Linear(4 * emb_dim, emb_dim)` — 원래 차원으로 복구

**왜 4배?**: 원 Transformer 논문/GPT-2의 경험적 선택. 너무 좁으면 표현력 부족, 너무 넓으면 비용 폭증.

**GELU**: `0.5x(1 + tanh(√(2/π)(x + 0.044715x³)))` — 근사식. x가 음수여도 작은 값 전달(ReLU는 0).

### 문제 8. FeedForward (GPT-style MLP)

In [ ]:
# [코드]
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], ### 4 ### 작성 필요 ### * cfg["emb_dim"]),
            ### GELU ### 작성 필요 ###(),
            nn.Linear(### 4 ### 작성 필요 ### * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

## 핵심 개념 9. Transformer Block (Pre-LN + Residual)

**두 개의 서브 블록** (각각 Residual connection):
1. Attention Block: `x = x + Dropout(Attention(LN(x)))`
2. FeedForward Block: `x = x + Dropout(FF(LN(x)))`

**Pre-LN 구조 (GPT-2)**: LayerNorm을 서브블록 "앞"에 적용 → 학습 안정성 향상 (vs. 원 Transformer의 Post-LN).

**Residual (`x + shortcut`)**: 입력을 그대로 더해주는 skip connection.
- 깊은 네트워크에서 gradient가 사라지는 것 방지.
- shortcut이 없으면 12~48층 짜리 GPT는 학습 불가.

**비워진 부분**: `self.att(x)` (attention 호출), `self.ff(x)` (feedforward 호출).

### 문제 9. Transformer Block forward (Residual 연결 포함)

In [ ]:
# [코드]
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.### att ### 작성 필요 ###(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.### ff ### 작성 필요 ###(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

## 핵심 개념 10. GPTModel (전체 아키텍처)

**구조**:
```
Token Emb + Pos Emb → Dropout → N × TransformerBlock → LayerNorm → Linear (out_head) → logits
```

**__init__ 인자 매핑**:
- `tok_emb = nn.Embedding(vocab_size, emb_dim)` — 어휘 크기만큼의 embedding table.
- `pos_emb = nn.Embedding(context_length, emb_dim)` — 위치 개수만큼. **vocab_size 아님 주의**.
- `out_head = nn.Linear(emb_dim, vocab_size)` — 각 토큰에 대해 다음 단어 logit(vocab_size 개) 출력.

**forward 흐름**:
1. `torch.arange(seq_len, device=in_idx.device)` → `[0, 1, ..., seq_len-1]` 로 위치 인덱스 생성.
2. `tok_embeds + pos_embeds` 합산 (broadcast).
3. `drop_emb` → `trf_blocks`(Sequential로 N층 통과) → `final_norm` → `out_head`.
4. 출력 `logits` shape: `(batch, seq_len, vocab_size)` — 각 위치마다 다음 토큰 분포.

**bias=False in out_head**: 임베딩과 output 가중치를 공유(weight tying)하는 경우 bias 없이. 여기선 공유는 안 하지만 관례상 False.

### 문제 10. GPTModel (전체 아키텍처 조립 + forward)

In [ ]:
# [코드]
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["### vocab_size ### 작성 필요 ###"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["### context_length ### 작성 필요 ###"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["### vocab_size ### 작성 필요 ###"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(### torch.arange(seq_len, device=in_idx.device) ### 작성 필요 ###)

        x = tok_embeds + ### pos_embeds ### 작성 필요 ###
        x = self.drop_emb(x)
        x = self.### trf_blocks ### 작성 필요 ###(x)
        x = self.final_norm(x)
        logits = self.### out_head ### 작성 필요 ###(x)

        return logits

## 핵심 개념 11. Greedy Text Generation

**Auto-regressive 생성**: 모델이 예측한 토큰을 다시 입력에 이어붙여 다음 토큰을 예측 → 반복.

**단계별**:
1. `idx_cond = idx[:, -context_size:]` — 모델의 최대 컨텍스트 길이를 넘지 않도록 슬라이딩.
2. `with torch.no_grad():` — **추론 중에는 gradient 불필요** → 메모리 절약 + 속도 향상.
3. `logits[:, -1, :]` — 시퀀스의 **마지막 토큰 logit만** 사용. 다음 토큰 예측이므로.
4. `torch.argmax(logits, dim=-1, keepdim=True)` — **Greedy decoding**: 가장 확률 높은 토큰 선택. `keepdim=True`로 shape `(batch, 1)` 유지.
5. `torch.cat((idx, idx_next), dim=1)` — 시퀀스 뒤에 새 토큰 append.

**Greedy의 한계**: 항상 최고 확률만 고르므로 반복적/단조로운 출력. 다양성을 원하면 temperature sampling, top-k, top-p 등 필요.

### 문제 11. Greedy Text Generation Loop

In [ ]:
# [코드]
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.### no_grad ### 작성 필요 ###():
            logits = model(idx_cond)
        logits = logits[:, ### -1 ### 작성 필요 ###, :]

        idx_next = torch.### argmax ### 작성 필요 ###(logits, dim=-1, keepdim=True)
        idx = torch.### cat ### 작성 필요 ###((idx, idx_next), dim=1)

    return idx

## 핵심 개념 12. Pretraining Loss (Cross Entropy)

**손실 함수**: 각 위치에서 다음 토큰을 vocab 전체에 대한 분류 문제로 풀기 → Cross Entropy.

**텐서 shape 맞추기 (중요!)**:
- `logits` shape: `(batch, seq_len, vocab_size)` — 3D
- `target` shape: `(batch, seq_len)` — 2D (정수 class index)
- `F.cross_entropy`는 `input=(N, C)` + `target=(N,)` 형태 기대.
- 해결: **모든 배치·시퀀스를 한 덩어리로 flatten**
  - `logits.flatten(0, 1)` → `(batch*seq_len, vocab_size)`
  - `target_batch.flatten()` → `(batch*seq_len,)`

**Cross Entropy 동작**: `−log(softmax(logits)[target])` — 정답 토큰의 log-확률에 음수 붙임. PyTorch는 log_softmax + NLL을 한 번에 계산(수치 안정성).

### 문제 12. Pretraining Loss (Cross Entropy over Vocab)

In [ ]:
# [코드]
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.### cross_entropy ### 작성 필요 ###(
        logits.### flatten ### 작성 필요 ###(0, 1),
        target_batch.### flatten ### 작성 필요 ###()
    )
    return loss

## 핵심 개념 13. PyTorch 표준 학습 루프

**5단계 (순서 중요!)**:
1. `model.train()` — **훈련 모드** 전환 (Dropout/BatchNorm 활성화). 평가 전에는 `model.eval()`.
2. `optimizer.zero_grad()` — 이전 배치 gradient 초기화. **PyTorch는 gradient를 누적**하므로 매 iteration 초기화 필수.
3. `loss = calc_loss_batch(...)` — 순전파 + loss 계산.
4. `loss.backward()` — 역전파로 모든 파라미터의 gradient 계산.
5. `optimizer.step()` — gradient를 사용해 파라미터 업데이트.

**순서 주의**: `zero_grad → forward → loss → backward → step`. `backward` 이전에 `zero_grad`. `backward` 이후에 `step`.

**model.train() vs model.eval()**: train 모드는 Dropout 활성, BatchNorm running stats 업데이트. 평가 시 eval 모드로 바꿔야 결정적이고 올바른 추론.

### 문제 13. PyTorch 학습 루프 (모드 전환 + zero_grad / backward / step)

In [ ]:
# [코드]
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.### train ### 작성 필요 ###()

        for input_batch, target_batch in train_loader:
            optimizer.### zero_grad ### 작성 필요 ###()

            loss = ### calc_loss_batch ### 작성 필요 ###(input_batch, target_batch, model, device)

            loss.### backward ### 작성 필요 ###()

            optimizer.### step ### 작성 필요 ###()

            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)

    return train_losses, val_losses, track_tokens_seen

## 핵심 개념 14. Text ↔ Token IDs 변환

**문제**: 모델은 `(batch, seq_len)` 형태의 텐서를 기대하지만 단일 텍스트는 1D 리스트가 됨.

- `text → token_ids`:
  1. `tokenizer.encode(text)` → Python list `[id1, id2, ...]`
  2. `torch.tensor(encoded)` → 1D tensor shape `(seq_len,)`
  3. `.unsqueeze(0)` → **batch 차원 추가** → `(1, seq_len)` ← 모델 입력 형태

- `token_ids → text`:
  1. `token_ids.squeeze(0)` → batch 차원 제거 → `(seq_len,)`
  2. `.tolist()` → Python list
  3. `tokenizer.decode(...)` → 문자열

**unsqueeze(dim) vs squeeze(dim)**: 차원 추가 vs 제거. dim=0은 맨 앞(batch) 축.

### 문제 14. Text ↔ Token IDs 변환 (batch 차원 추가/제거)

In [ ]:
# [코드]
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).### unsqueeze ### 작성 필요 ###(0)
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.### squeeze ### 작성 필요 ###(0)
    return tokenizer.decode(flat.tolist())

## 핵심 개념 15. LoRA (Low-Rank Adaptation)

**아이디어**: 거대 모델의 모든 파라미터를 fine-tuning 하지 말고, 기존 Linear에 **저차원(rank r)의 보조 경로**만 추가해 그것만 학습.

**수식**: 기존 `y = Wx`에 LoRA 보정 추가 → `y = Wx + (α/r) · B·A·x`
- `A ∈ ℝ^(in_dim × r)` : down-projection (Kaiming uniform 초기화 — 랜덤)
- `B ∈ ℝ^(r × out_dim)` : up-projection (**0으로 초기화** → 학습 시작 시점에는 LoRA 기여 0 → 원래 출력 보존)
- `α/r`: 스케일링 계수. rank를 바꿔도 업데이트 크기가 일정하도록.

**왜 B=0?**: 학습 초반에 원 모델을 망치지 않기 위해. A는 랜덤이어도 B·(A·x) = 0이라 기여 없음.

**LinearWithLoRA**: `linear(x) + lora(x)` — **원래 Linear는 유지**하고 LoRA 출력을 더함.

**replace_linear_with_lora**: 모델을 **재귀 탐색**하며 `torch.nn.Linear` 타입을 `LinearWithLoRA`로 교체. `named_children()`은 직속 자식만 순회 → 재귀 필요.

**장점**: 학습 파라미터 수가 full fine-tuning의 0.1~1% 수준이면서 성능은 비슷. 여러 task별 LoRA 가중치만 저장/교체 가능.

### 문제 15. LoRA (Low-Rank Adaptation) 레이어

In [ ]:
# [코드]
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(in_dim, ### rank ### 작성 필요 ###))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

        self.B = nn.Parameter(torch.zeros(### rank ### 작성 필요 ###, out_dim))

        self.alpha = alpha
        self.rank = rank

    def forward(self, x):
        x = (self.### alpha ### 작성 필요 ### / self.rank) * (x @ self.### A ### 작성 필요 ### @ self.### B ### 작성 필요 ###)
        return x


class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)

    def forward(self, x):
        return self.linear(x) + self.### lora ### 작성 필요 ###(x)


def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.### Linear ### 작성 필요 ###):
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            replace_linear_with_lora(module, rank, alpha)

## 핵심 개념 16. LoRA 적용 절차 (Freeze → Replace)

**LoRA fine-tuning 2단계**:
1. **모든 기존 파라미터 freeze**: `param.requires_grad = False` → gradient 계산/업데이트 차단.
2. **Linear를 LinearWithLoRA로 교체**: `replace_linear_with_lora(model, rank, alpha)` 호출.
   - 교체 시 새로 추가된 LoRA의 A, B는 **자동으로 `requires_grad=True`** (기본값).
   - 따라서 오직 LoRA 파라미터만 학습됨.

**검증**: `sum(p.numel() for p in model.parameters() if p.requires_grad)` — 수백만/수십억 → 몇십만으로 급감하면 성공.

### 문제 16. LoRA 적용 (전체 freeze → LoRA만 학습)

In [ ]:
# [코드]
for param in model.parameters():
    param.requires_grad = ### False ### 작성 필요 ###

### replace_linear_with_lora ### 작성 필요 ###(model, rank=LORA_RANK, alpha=LORA_ALPHA)

model.to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"총 학습 가능 LoRA 파라미터 수: {total_params:,}")

## 핵심 개념 17. 분류 Fine-tuning (Spam 분류 등)

**GPT를 분류기로 변환**:
1. **전체 freeze**: `requires_grad = False` — pretrained 가중치 보존.
2. **출력 헤드 교체**: `out_head`를 `Linear(emb_dim, num_classes)`로 교체 → vocab 예측이 아닌 클래스 예측.
3. **마지막 블록 + final_norm만 unfreeze**: `model.trf_blocks[-1]`과 `final_norm`만 `requires_grad=True` → 적은 파라미터로 빠른 fine-tuning.

**손실 계산 시 주의**: GPT 출력은 `(batch, seq_len, num_classes)` — 시퀀스의 **마지막 토큰** 예측만 사용. 이유: 시퀀스 전체의 맥락을 누적해서 본 마지막 위치가 요약 정보를 담고 있음.
- `logits = model(input_batch)[:, -1, :]` → shape `(batch, num_classes)`

**왜 마지막 블록만?**: Pretrained은 low-level은 일반 언어 패턴, high-level은 task 특화 → 상위 몇 층만 학습해도 충분.

### 문제 17. 분류 Fine-tuning (마지막 토큰 logit + 선택적 freeze)

In [ ]:
# [코드]
for param in model.parameters():
    param.requires_grad = ### False ### 작성 필요 ###

torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=### num_classes ### 작성 필요 ###)

model.to(device)

for param in model.trf_blocks[### -1 ### 작성 필요 ###].parameters():
    param.requires_grad = ### True ### 작성 필요 ###

for param in model.final_norm.parameters():
    param.requires_grad = ### True ### 작성 필요 ###


def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, ### -1 ### 작성 필요 ###, :]
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

## 핵심 개념 18. Instruction Finetuning Collate (Padding + Shift + Ignore)

**가변 길이 시퀀스 → 고정 길이 배치 텐서 만드는 과정**:

1. **EOS 토큰 추가**: 각 샘플 뒤에 `pad_token_id(=50256, <|endoftext|>)` 하나 붙임 → 문장 종료 표시.
2. **Padding**: 배치 최대 길이에 맞춰 `pad_token_id`로 뒤를 채움.
3. **Next-token shift**:
   - `inputs = padded[:-1]` — 마지막 토큰 제외한 앞부분.
   - `targets = padded[1:]` — 첫 토큰 제외한 뒷부분.
   - 이렇게 해야 position i의 입력에 대해 position i+1이 정답.
4. **Ignore index 처리**: padding 구간에서는 loss를 계산하지 않도록 `targets`에서 **두 번째 pad 토큰부터** `ignore_index(=-100)`로 변환.
   - `indices[1:]` — 첫 번째 pad는 "여기서 문장 끝" 신호로 학습에 유효하므로 남기고, 나머지는 무시.
   - `F.cross_entropy`는 target이 `-100`인 위치를 자동 제외.

**왜 첫 pad는 유지?**: 모델이 "언제 문장을 끝내야 하는지"를 학습해야 하므로 EOS 예측은 필요. 나머지 padding은 진짜 데이터가 아니므로 무시.

### 문제 18. Instruction Finetuning용 Collate 함수 (Shift + Padding + Mask)

In [ ]:
# [코드]
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    batch_max_length = max(len(item)+1 for item in batch)

    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [### pad_token_id ### 작성 필요 ###]

        padded = new_item + [### pad_token_id ### 작성 필요 ###] * (batch_max_length - len(new_item))

        inputs = torch.tensor(padded[:### -1 ### 작성 필요 ###])
        targets = torch.tensor(padded[### 1 ### 작성 필요 ###:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[### 1 ### 작성 필요 ###:]] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

## 핵심 개념 19. DPO (Direct Preference Optimization)

**DPO**: 인간 선호 데이터 `(prompt, chosen, rejected)`로 RLHF 없이 **직접** 모델을 정렬(alignment)하는 방법. Reference 모델 대비 policy 모델이 chosen은 더 선호하고 rejected는 덜 선호하도록 학습.

### compute_logprobs: 정답 토큰의 로그 확률

**Auto-regressive Shift** (next-token prediction의 본질):
- 모델은 `[t0, t1, ..., t_{n-1}]` 입력 → `[t1, t2, ..., t_n]` 예측.
- 따라서 정답(labels)은 **1칸 앞으로**: `labels[:, 1:]`
- logits는 **마지막 시점 제외**: `logits[:, :-1, :]`
- 이제 shape는 `labels=(b, n-1)`, `logits=(b, n-1, V)` 로 정렬.

**torch.gather로 정답 토큰 확률만 추출**:
- `log_probs = F.log_softmax(logits, dim=-1)` — 모든 vocab에 대한 log prob.
- `torch.gather(input=log_probs, dim=-1, index=labels.unsqueeze(-1))` — 각 위치에서 labels이 가리키는 token의 log prob만 pick.
- **`input=log_probs`** (raw logits 아님!): 확률 공간에서 뽑아야 의미가 있음.

### compute_dpo_loss: 선호-기반 손실

**수식**:

$$ \mathcal{L}_{DPO} = -\mathbb{E}\left[\log \sigma\left(\beta \cdot \left( (\log\pi_\theta(y_w|x) - \log\pi_\theta(y_l|x)) - (\log\pi_{ref}(y_w|x) - \log\pi_{ref}(y_l|x)) \right) \right)\right] $$

- Policy의 chosen-rejected 차이가 reference의 차이보다 **커질수록** loss 감소.
- `F.logsigmoid(x)`: `log(σ(x))` — 수치 안정적. 음수 붙이면 NLL.
- `β` (beta): 0.1~0.5 — reference에서 얼마나 벗어날지 규제 세기.

### 문제 19. DPO Log Probability & Loss

In [ ]:
# [코드]
def compute_logprobs(logits, labels, selection_mask=None):
    labels = labels[:, ### 1 ### 작성 필요 ###:].clone()
    logits = logits[:, :### -1 ### 작성 필요 ###, :]

    log_probs = F.log_softmax(logits, dim=-1)

    selected_log_probs = torch.gather(
        input=### log_probs ### 작성 필요 ###,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    if selection_mask is not None:
        mask = selection_mask[:, 1:].clone()
        selected_log_probs = selected_log_probs * mask
        avg_log_prob = selected_log_probs.sum(-1) / mask.sum(-1)
        return avg_log_prob
    else:
        return selected_log_probs.mean(-1)


def compute_dpo_loss(model_chosen_logprobs, model_rejected_logprobs,
                     reference_chosen_logprobs, reference_rejected_logprobs, beta=0.1):
    model_logratios = model_chosen_logprobs - model_rejected_logprobs
    reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs

    logits = model_logratios - reference_logratios

    losses = -F.### logsigmoid ### 작성 필요 ###(beta * logits)

    chosen_rewards = (model_chosen_logprobs - reference_chosen_logprobs).detach()
    rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs).detach()

    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()